# Task
Create a small search or recommendation system that matches a user query with the most relevant text documents using TF-IDF and cosine similarity.
 
Example Scenario
You have 10 movie descriptions. A user searches: “space adventure with robots” 
Your system should recommend the movie descriptions that are most similar to this query
 
- Create a dataset
- Use at least 10 short text documents
- Each document should have a title and description
- Preprocess the text
- Lowercase
- Remove punctuation
- Remove stopwords
- Lemmatize or stem words
- Convert text into TF-IDF vectors
- Use tf-idf vectorizer from sklearn
- Accept a user query
- Example: "healthy food"
- Preprocess the query in the same way as the documents
- Calculate cosine similarity
- Compare the query vector with all document vectors
- Return the top 3 results
- Show title
- Show description
- Show similarity score

In [1]:
import pandas as pd
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Dataset

In [2]:
df = pd.read_csv('movies.csv', encoding='utf-8', sep=";")

print(f"Movie database containing the following:\n")
print(df[['title', 'description']].to_string(index=False, justify='left'))


Movie database containing the following:

title                      description                                                                                                                                                                                                                                                                                        
                Space Wars A young farm boy discovers his destiny as a space warrior. Accompanied by an old knight, a loyal robot droid, and a courageous rebel, he journeys through space adventures and fights against a tyrannical empire. Spectacular space battles and the power of the Jedi define this epic space saga.
 Welcome to the real world           A computer hacker discovers a terrifying truth: the reality he knows is a simulated virtual world. Machines and robots have enslaved humanity and use it as an energy source. As the chosen one, he fights against the machine world and must overcome the limitations of the simulation.
 

## Preprocess the text
- Lowercase
- Remove punctuation
- Remove stopwords
- Lemmatize or stem words
- tidy up

In [10]:
nlp = spacy.load("en_core_web_sm")

print(f"Stop words {sorted(list(nlp.Defaults.stop_words))}\nIn total: {len(nlp.Defaults.stop_words)}")

def preprocess(text):

    doc = nlp(text)

    tokens = [
        token.lemma_.lower()
        for token in doc
        if not token.is_stop and token.is_alpha
    ]

    # Compose tokens for TF-IDF
    return ' '.join(tokens)

# Additional field for the adjusted text for comparison
results = []
for text in df['description']:
    results.append(preprocess(text))
df['preprocessed'] = results

print("\nComparison Original vs. Pre-processed:\n")
for _, row in df.iterrows():
    print(f"Movie: {row['title']}")
    print(f"- Original:    {row['description'][:70]}...{row['description'][-30:]}")
    print(f"- Processed:   {row['preprocessed'][:70]}...{row['preprocessed'][-30:]}\n")


Stop words ["'d", "'ll", "'m", "'re", "'s", "'ve", 'a', 'about', 'above', 'across', 'after', 'afterwards', 'again', 'against', 'all', 'almost', 'alone', 'along', 'already', 'also', 'although', 'always', 'am', 'among', 'amongst', 'amount', 'an', 'and', 'another', 'any', 'anyhow', 'anyone', 'anything', 'anyway', 'anywhere', 'are', 'around', 'as', 'at', 'back', 'be', 'became', 'because', 'become', 'becomes', 'becoming', 'been', 'before', 'beforehand', 'behind', 'being', 'below', 'beside', 'besides', 'between', 'beyond', 'both', 'bottom', 'but', 'by', 'ca', 'call', 'can', 'cannot', 'could', 'did', 'do', 'does', 'doing', 'done', 'down', 'due', 'during', 'each', 'eight', 'either', 'eleven', 'else', 'elsewhere', 'empty', 'enough', 'even', 'ever', 'every', 'everyone', 'everything', 'everywhere', 'except', 'few', 'fifteen', 'fifty', 'first', 'five', 'for', 'former', 'formerly', 'forty', 'four', 'from', 'front', 'full', 'further', 'get', 'give', 'go', 'had', 'has', 'have', 'he', 'hence', 'her', 

## Convert text into TF-IDF vectors
Use tf-idf vectorizer from sklearn

In [4]:
vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(df['preprocessed'])

# shape[1] returns the number of dimensions or features (unique words)
print(f"Each film is represented by a vector with {tfidf_matrix.shape[1]} dimensions described.")

Each film is represented by a vector with 263 dimensions described.


In [11]:
feature_names = vectorizer.get_feature_names_out()
tfidf_array = tfidf_matrix.toarray()

# Select film title for analysis
i = df[df['title'] == "Space Wars"].index[0]

# Array corresponds to the film’s vector, which contains all TF-IDF values for all words, sorted in descending order at the end.
column_index = tfidf_array[i].argsort()[::-1]

terms = []
for j in column_index:
    word = feature_names[j]
    value = tfidf_array[i][j]
    terms.append(f"{word} ({value:.3f})")

print(f"{df['title'][i]}:")
print(f"TF-IDF Values: {', '.join(terms)}")
print("High TF-IDF score = the word is significant for this film (it appears frequently in this film, but rarely in others)")

Space Wars:
TF-IDF Values: space (0.584), spectacular (0.170), saga (0.170), tyrannical (0.170), rebel (0.170), old (0.170), loyal (0.170), knight (0.170), journey (0.170), jedi (0.170), farm (0.170), accompany (0.170), empire (0.170), droid (0.170), warrior (0.170), define (0.170), epic (0.170), discover (0.146), destiny (0.146), courageous (0.146), power (0.146), boy (0.146), battle (0.146), robot (0.146), adventure (0.146), young (0.116), fight (0.105), espionage (0.000), father (0.000), vast (0.000), fantasy (0.000), famous (0.000), family (0.000), fall (0.000), fail (0.000), face (0.000), experience (0.000), exile (0.000), exceptionally (0.000), evil (0.000), travel (0.000), flee (0.000), velociraptor (0.000), enslave (0.000), energy (0.000), victim (0.000), embark (0.000), elf (0.000), eccentric (0.000), earth (0.000), dwarf (0.000), virtual (0.000), dream (0.000), drama (0.000), escape (0.000), food (0.000), island (0.000), help (0.000), infiltrate (0.000), industrial (0.000), i

## Accept a user query
- Preprocess the query in the same way as the documents
- Calculate cosine similarity
- Compare the query vector with all document vectors
- Return the top 3 results
- Show title
- Show description
- Show similarity score

In [6]:
def search(enquiry):

    # Refine the query as before with the dataset
    processed_request = preprocess(enquiry)

    # Convert the query to a TF-IDF vector as well
    query_vector = vectorizer.transform([processed_request])

    # Cosine similarity between the query and all film vectors
    similarities = cosine_similarity(query_vector, tfidf_matrix)[0]
    # Sort films in descending order by similarity
    top_indizes = similarities.argsort()[::-1]

    print(f"Search query:   '{enquiry}'")
    print(f"Pre-processed:  '{processed_request}'")
    print("=" * 40)

    # Go through all the films (starting with 1)
    for rank, index in enumerate(top_indizes[:3], 1):
        similarity = similarities[index]
        print(f"\nPlacement {rank}: {df['title'].iloc[index]}")
        print(f"Similarity:     {similarity:.4f} ({similarity * 100:.1f}%)")
        print(f"Description:    {df['description'].iloc[index]}")
        print("----------------------------------")


In [7]:
search("Space Robots")

Search query:   'Space Robots'
Pre-processed:  'space robots'

Placement 1: Space Wars
Similarity:     0.5845 (58.4%)
Description:    A young farm boy discovers his destiny as a space warrior. Accompanied by an old knight, a loyal robot droid, and a courageous rebel, he journeys through space adventures and fights against a tyrannical empire. Spectacular space battles and the power of the Jedi define this epic space saga.
----------------------------------

Placement 2: Beyond the Stars
Similarity:     0.1725 (17.2%)
Description:    Earth is slowly dying. A former astronaut and space pilot travels through a wormhole to distant galaxies and stars in search of a new home planet for humanity. He experiences time distortion near a black hole and struggles to maintain contact with his family.
----------------------------------

Placement 3: A journey on ice to ruin
Similarity:     0.0000 (0.0%)
Description:    On board the supposedly unsinkable luxury liner, a penniless young artist falls i

In [12]:
search("Dreams")

Search query:   'Dreams'
Pre-processed:  'dream'

Placement 1: Limbus
Similarity:     0.3400 (34.0%)
Description:    An industrial espionage specialist infiltrates the dreams and subconscious of his victims to steal secret information. His latest assignment isn't to steal an idea, but to plant one. He delves deeper into nested dream levels as he tries to return to his family.
----------------------------------

Placement 2: A master chef on four legs
Similarity:     0.1617 (16.2%)
Description:    An exceptionally talented rat dreams of becoming a famous chef and cooking healthy food. In the kitchens of a renowned Parisian restaurant, she creates delicious dishes. Together with a kitchen boy, she delights even the city's toughest restaurant critic with her tasty meals.
----------------------------------

Placement 3: A journey on ice to ruin
Similarity:     0.0000 (0.0%)
Description:    On board the supposedly unsinkable luxury liner, a penniless young artist falls in love with a wealth

In [14]:
search("A film featuring princesses and ice")

Search query:   'A film featuring princesses and ice'
Pre-processed:  'film feature princess ice'

Placement 1: The Ice Princess
Similarity:     0.4359 (43.6%)
Description:    A courageous princess embarks on a perilous adventure through a winter landscape of ice and snow to rescue her sister, who has escaped the kingdom using ice magic. With the help of a cheerful snowman and a brave miner, she battles the cold and loneliness.
----------------------------------

Placement 2: A journey on ice to ruin
Similarity:     0.0000 (0.0%)
Description:    On board the supposedly unsinkable luxury liner, a penniless young artist falls in love with a wealthy heiress. Their forbidden romance blossoms on the Atlantic before the ship collides with a massive iceberg and sinks into the icy depths of the ocean along with its passengers.
----------------------------------

Placement 3: The Rise
Similarity:     0.0000 (0.0%)
Description:    An unknown amateur boxer from Philadelphia receives an unexpected